In [3]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"C:/Plegma_Programming")
DATA_PATH = BASE_DIR / "PLEGMA_final_hourly_dataset.csv"
MODELS_BASE = BASE_DIR / "models"

OUTPUT_DIR = MODELS_BASE / "permutation_importance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
TARGET = "consumption_Wh"

TOP_N = 15

MODEL_CONFIG = {
    "rf": MODELS_BASE / "final_rf",
    "xgb": MODELS_BASE / "final_xgb",
    "lgbm": MODELS_BASE / "final_lgbm",
}

MODES = ["coldstart", "withhistory"]

# =========================
# HELPERS
# =========================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def prepare_dataset(df):
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    return df

def split_global(df, test_days=30):
    cutoff = df[TIME_COL].max() - pd.Timedelta(days=test_days)
    train = df[df[TIME_COL] <= cutoff].copy()
    test = df[df[TIME_COL] > cutoff].copy()
    return train, test

def add_lags(df):
    df = df.sort_values(["home_id", TIME_COL])
    g = df.groupby("home_id")[TARGET]

    df["lag_1h"] = g.shift(1)
    df["lag_24h"] = g.shift(24)
    df["lag_168h"] = g.shift(168)
    df["roll_mean_24h"] = g.shift(1).rolling(24).mean()
    df["roll_mean_168h"] = g.shift(1).rolling(168).mean()

    return df

def preprocess(df):
    df = df.copy()

    # One-hot encoding
    df = pd.get_dummies(df, drop_first=False)

    return df

def align_columns(X, columns):
    return X.reindex(columns=columns, fill_value=0)

def plot_importance(df, title, path):
    df_top = df.head(TOP_N).iloc[::-1]

    plt.figure(figsize=(10, 7))
    plt.barh(df_top["feature"], df_top["importance"])
    plt.title(title)
    plt.xlabel("Permutation Importance (Δ MAE)")
    plt.tight_layout()
    plt.savefig(path, dpi=300)
    plt.close()

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(DATA_PATH)
df = prepare_dataset(df)

# =========================
# MAIN LOOP
# =========================
all_results = []

for model_name, model_dir in MODEL_CONFIG.items():

    print("\n=========================")
    print(f"MODEL: {model_name.upper()}")
    print("=========================")

    for mode in MODES:

        print(f"\n--- Mode: {mode} ---")

        model_path = model_dir / f"{model_name}_{mode}.joblib"
        features_path = model_dir / f"{model_name}_{mode}_feature_columns.json"

        if not model_path.exists() or not features_path.exists():
            print("Missing artifacts")
            continue

        model = joblib.load(model_path)
        feature_cols = load_json(features_path)

        # =========================
        # DATA PREP
        # =========================
        df_model = df.copy()

        if mode == "withhistory":
            df_model = add_lags(df_model)
            df_model = df_model.dropna()

        train, test = split_global(df_model)

        X_test = preprocess(test)
        X_test = align_columns(X_test, feature_cols)

        y_test = test[TARGET].values

        # =========================
        # PERMUTATION IMPORTANCE
        # =========================
        result = permutation_importance(
            model,
            X_test,
            y_test,
            scoring="neg_mean_absolute_error",
            n_repeats=5,
            random_state=42,
            n_jobs=-1
        )

        importance = -result.importances_mean

        df_imp = pd.DataFrame({
            "feature": feature_cols,
            "importance": importance
        }).sort_values("importance", ascending=False).reset_index(drop=True)

        prefix = f"{model_name}_{mode}"

        # Save CSV
        csv_path = OUTPUT_DIR / f"{prefix}_perm_importance.csv"
        df_imp.to_csv(csv_path, index=False)

        # Plot
        plot_path = OUTPUT_DIR / f"{prefix}_perm_importance.png"
        plot_importance(df_imp, prefix, plot_path)

        print(f"Saved: {csv_path}")
        print(df_imp.head(10))

        df_temp = df_imp.copy()
        df_temp["model"] = model_name
        df_temp["mode"] = mode
        all_results.append(df_temp)

# =========================
# COMBINED FILE
# =========================
if all_results:
    df_all = pd.concat(all_results, ignore_index=True)
    df_all.to_csv(OUTPUT_DIR / "all_models_permutation_importance.csv", index=False)

print("\nDone.")


MODEL: RF

--- Mode: coldstart ---
Saved: C:\Plegma_Programming\models\permutation_importance\rf_coldstart_perm_importance.csv
                       feature  importance
0            internal_humidity    0.069246
1    years_in_house_gt_5_years    0.039119
2         external_temperature    0.037525
3            external_humidity    0.029311
4                         hour    0.018454
5          build_era_1950_1970    0.015827
6          has_washing_machine    0.015644
7     income_band_up_to_1_wage    0.014962
8  occupation_Master's Student    0.014012
9         internal_temperature    0.012095

--- Mode: withhistory ---
Saved: C:\Plegma_Programming\models\permutation_importance\rf_withhistory_perm_importance.csv
                       feature  importance
0                       lag_1h    0.037613
1    years_in_house_gt_5_years    0.016261
2                roll_mean_24h    0.015969
3                      lag_24h    0.006844
4                         hour    0.006264
5          build_era